<div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 60%, #0c4a6e 100%); padding: 28px 30px; border-radius: 10px; border-bottom: 4px solid #38bdf8; margin-bottom: 16px;">
    <h1 style="margin: 0; color: #ffffff; font-size: 28px; font-weight: 700; letter-spacing: 0.4px;">
        Employee Attrition Prediction: EDA &amp; Multi-Model Benchmarking
    </h1>
    <p style="margin: 10px 0 0 0; color: #cbd5e1; font-size: 15px; line-height: 1.6;">
        An end-to-end walkthrough of exploratory data analysis, leak-free preprocessing pipelines, handling severe class imbalance with SMOTENC, and comparing <strong>KNN</strong>, <strong>Random Forest</strong>, and <strong>Logistic Regression</strong> on HR attrition data.
    </p>
</div>

### Overview

Predicting employee attrition sounds straightforward on paper, but HR datasets tend to be messy in subtle ways—mostly because the vast majority of employees stay, while the minority class we actually care about (employees leaving) is scattered across a lot of weakly correlated features.

In this notebook, I walk through the full workflow step by step: inspecting the raw data, splitting *before* doing any visual EDA to avoid peeking at the test set, building reusable preprocessing pipelines, testing a baseline KNN model, diagnosing why it struggles, and then expanding the experiment with **SMOTENC**, **Random Forest**, and **Logistic Regression** to see how different algorithms respond to the imbalance.

---

### Table of Contents

1. [Loading & Inspecting the Dataset](#section-1)
2. [Stratified Train-Test Split](#section-2)
3. [Exploratory Data Analysis (EDA) & Visual Patterns](#section-3)
4. [Building the Preprocessing & Training Pipeline](#section-4)
5. [Baseline Model: K-Nearest Neighbors (KNN)](#section-5)
6. [Hyperparameter Tuning & Test Evaluation (Baseline KNN)](#section-6)
7. [Diagnosing Baseline Performance](#section-7)
8. [Tackling Class Imbalance with SMOTENC (KNN)](#section-8)
9. [Model Comparison: Random Forest Classifier](#section-9)
10. [Model Comparison: Logistic Regression](#section-10)
11. [Final Takeaways](#section-11)

<a id="section-1"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">01</span> &nbsp;|&nbsp; Loading &amp; Inspecting the Dataset
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Loading the raw CSV, checking dimensions, data types, missing values, and categorical cardinality.</p>
</div>

First, I'll load the dataset from CSV and take a quick look at the top and bottom rows along with the schema to get a feel for the dimensions and overall quality of the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df_raw = pd.read_csv("data/emp.csv")
df_raw.head()

In [ ]:
df_raw.tail()

In [ ]:
df_raw.info()

#### Missing Value Audit

Checking the number of null and missing values in each column before deciding on imputation or cleaning steps.

In [ ]:
df_raw.isna().sum()

In [ ]:
df_raw.isnull().sum()

There are no missing values in the raw CSV, which is nice to see—though I will still include imputers in the pipeline later so the workflow stays resilient if unseen data has nulls.

#### Inspecting Categorical Features

Next, I'll check which text columns are categorical and how many unique values each one has. This helps determine whether One-Hot Encoding makes sense or if Label/Ordinal Encoding is more practical.

In [ ]:
list_of_text_cols = df_raw.select_dtypes(include="str").columns.tolist()

for col in list_of_text_cols:
    print(df_raw[col].value_counts() , "\n")

<div style="border-left: 4px solid #38bdf8; background-color: rgba(56, 189, 248, 0.08); padding: 12px 16px; border-radius: 0 6px 6px 0; margin: 10px 0;">
    <strong>Feature Encoding &amp; Pruning Plan</strong><br>
    Based on the value counts and schema above:<br>
    &bull; <strong>Drop zero-variance / ID columns:</strong> <code>Over18</code>, <code>StandardHours</code>, <code>EmployeeNumber</code>, and <code>EmployeeCount</code> carry no predictive signal (either constant across all rows or just a unique ID), so dropping them right away reduces unnecessary compute.<br>
    &bull; <strong>Ordinal / Label Encoding candidates:</strong> <code>BusinessTravel</code> and <code>MaritalStatus</code> have related categories. <code>JobRole</code> has 9 categories, so label encoding keeps dimensionality down in the baseline (though I'll also test One-Hot Encoding later to see if the extra dimensions and compute are worth it).<br>
    &bull; <strong>One-Hot Encoding candidates:</strong> The remaining nominal columns (<code>OverTime</code>, <code>Gender</code>, <code>Department</code>, <code>EducationField</code>) have low cardinality and can be cleanly one-hot encoded.<br>
    &bull; <strong>Class imbalance spotted:</strong> <code>Attrition</code> has 1,233 <em>No</em> vs. 237 <em>Yes</em> (~5.2:1 ratio), which will definitely shape how I split and evaluate the models.
</div>

In [ ]:
df_raw = df_raw.drop(columns=["Over18", "StandardHours", "EmployeeNumber", "EmployeeCount"])

In [ ]:
df_raw.info()

<a id="section-2"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">02</span> &nbsp;|&nbsp; Stratified Train-Test Split
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Isolating the test set before visual EDA to prevent data leakage and optimistic bias.</p>
</div>

I'm splitting the data *before* checking linear correlations or visual non-linear patterns. Exploring the entire dataset before splitting can quietly introduce selection bias and optimistic bias into modeling decisions.

Since `Attrition` is heavily imbalanced, I'm using a **stratified 80/20 split** so both the training and test sets preserve the exact same target class ratio.

In [ ]:
from sklearn.model_selection import train_test_split

df_train : pd.DataFrame
df_test : pd.DataFrame

df_train, df_test = train_test_split(df_raw, test_size=0.2, stratify=df_raw["Attrition"], random_state=42)

Let's verify that the majority-to-minority class ratio is consistent across both splits.

In [ ]:
df_train["Attrition"].value_counts()

In [ ]:
986/190

In [ ]:
df_test["Attrition"].value_counts()

In [ ]:
247/47

In [ ]:
df_train.head()

Both sets sit right around a **5.2:1** (`No` to `Yes`) ratio, so the stratified split worked as intended. From here on, all EDA and hyperparameter tuning will happen strictly on `df_train`.

<a id="section-3"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">03</span> &nbsp;|&nbsp; Exploratory Data Analysis (EDA) &amp; Visual Patterns
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Examining feature distributions, target imbalance, and how numerical and categorical variables relate to attrition.</p>
</div>

I'll start with a broad look at the numerical feature distributions and pairwise relationships to spot skewness, outliers, or obvious cluster separation between employees who stay and those who leave.

In [ ]:
df_train.hist(bins=50, figsize=(20,20))

In [ ]:
import seaborn as sns

sns.pairplot(df_train, hue="Attrition")

In [ ]:
sns.countplot(df_train, x="Attrition", hue="Attrition")

The full pairplot is pretty dense, so let's zoom in on a smaller subset of key continuous features (`Age`, `MonthlyIncome`, `DistanceFromHome`) colored by `Attrition`.

In [ ]:
sns.pairplot(
    df_train[["Age", "MonthlyIncome", "DistanceFromHome", "Attrition"]],
    hue="Attrition",
    palette="plasma"
)

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">3.1 Numerical Features vs. Attrition</h3>
</div>

Now I'll inspect individual numerical features against `Attrition` using boxplots and normalized density histograms (`common_norm=False`). Using density normalization is important here because raw counts would just be drowned out by the majority `No` class.

In [ ]:
sns.boxplot(
    data=df_train,
    x="Attrition",
    y="Age",
    hue="Attrition",
)

plt.show()

In [ ]:
sns.histplot(x="DistanceFromHome", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.boxplot(x="Attrition", y="EnvironmentSatisfaction", hue="Attrition", data=df_train)
plt.show()

In [ ]:
sns.histplot(x="HourlyRate", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.histplot(x="JobSatisfaction", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.histplot(x="JobInvolvement", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.boxplot(x="Attrition", y="MonthlyIncome", data=df_train, hue="Attrition")
plt.plot()

In [ ]:
sns.histplot(x="MonthlyIncome", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.histplot(x="YearsAtCompany", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.boxplot(x="Attrition", y="YearsAtCompany", data=df_train, hue="Attrition")

In [ ]:
sns.histplot(x="YearsInCurrentRole", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.boxplot(x="Attrition", y="YearsInCurrentRole", data=df_train, hue="Attrition",)

In [ ]:
sns.histplot(x="YearsSinceLastPromotion", data=df_train, hue="Attrition", common_norm=False, stat="density")

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">3.2 Categorical Features vs. Attrition</h3>
</div>

Next, I'll check how attrition rates break down across departments, travel frequency, roles, marital status, gender, and overtime using row-normalized crosstabs (`normalize="index"`).

In [ ]:
df_train.info()

In [ ]:
pd.crosstab(df_train["Department"], df_train["Attrition"], normalize="index").plot(kind="bar", stacked=True)
plt.plot()

In [ ]:
sns.countplot(x="Department", hue="Attrition", data=df_train,)

In [ ]:
pd.crosstab(df_train["Gender"], df_train["Attrition"], normalize="index").plot(kind="bar")

In [ ]:
pd.crosstab(df_train["BusinessTravel"], df_train["Attrition"], normalize="index").plot(kind="bar")
plt.plot()

In [ ]:
pd.crosstab(df_train["JobRole"], df_train["Attrition"], normalize="index").plot(kind="bar")
plt.plot()

In [ ]:
pd.crosstab(df_train["MaritalStatus"], df_train["Attrition"], normalize="index").plot(kind="bar")

In [ ]:
pd.crosstab(df_train["OverTime"], df_train["Attrition"], normalize="index").plot(kind="bar",)
plt.plot()

<div style="border-left: 4px solid #10b981; background-color: rgba(56, 189, 248, 0.08); padding: 12px 16px; border-radius: 0 6px 6px 0; margin: 10px 0;">
    <strong>EDA Summary</strong><br>
    Exploring these numerical and categorical features shows that most of them carry at least some signal around attrition (for example, younger employees, lower monthly income, frequent travel, single marital status, and working overtime all lean noticeably toward higher attrition). Because almost every remaining feature contributes useful information, I won't drop anything else beyond the zero-variance and ID columns already removed earlier (<code>Over18</code>, <code>StandardHours</code>, <code>EmployeeNumber</code>, and <code>EmployeeCount</code>).<br><br>
    At the same time, there is heavy overlap between the two classes across almost every plot—no single feature cleanly separates leavers from stayers.
</div>

<a id="section-4"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">04</span> &nbsp;|&nbsp; Building the Preprocessing &amp; Training Pipeline
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Encoding the target, separating feature types, and assembling a leak-free ColumnTransformer.</p>
</div>

Here I'll encode the target variable (`Attrition`) into binary labels and build a single preprocessing pipeline with imputation, scaling, and encoding. Wrapping everything inside a `ColumnTransformer` and `Pipeline` keeps the workflow clean, prevents data leakage during cross-validation, and makes it easy to swap models across experiments.

*Note: Since KNN is distance-based, scaling numerical features with `StandardScaler` is essential so high-magnitude columns like `MonthlyIncome` don't dominate the distance calculations.*

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder =LabelEncoder()

df_train_x = df_train.drop(columns=["Attrition"])
df_train_y = np.asarray(label_encoder.fit_transform(df_train["Attrition"]))

df_test_x = df_test.drop(columns=["Attrition"])
df_test_y = np.asarray(label_encoder.transform(df_test["Attrition"]))



numerical_cols = df_train.select_dtypes(include="int64").columns.tolist()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder

simple_imputer_num = SimpleImputer(strategy="median")
simple_imputer_cat = SimpleImputer(strategy="most_frequent")
standard_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(sparse_output=False)
ordinal_encoder = OrdinalEncoder()

cat_onehot_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat), ("oneHotEncoder", one_hot_encoder)])
cat_label_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat), ("labelEncoder", ordinal_encoder)])
num_pipeline = Pipeline([("simpleImputerNum", simple_imputer_num), ("standardScaler", standard_scaler)])

final_model = ColumnTransformer([("catLabel", cat_label_pipeline, ["BusinessTravel" ,"JobRole", "MaritalStatus"]),
                                 ("catOneHot", cat_onehot_pipeline, ["OverTime", "Gender", "Department", "EducationField"]),
                                 ("num", num_pipeline, numerical_cols)])

<a id="section-5"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">05</span> &nbsp;|&nbsp; Baseline Model: K-Nearest Neighbors (KNN)
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Sweeping K values with F1 error curves, running 5-fold cross-validation, and checking test performance.</p>
</div>

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">5.1 Finding the Optimal Number of Neighbors (K)</h3>
</div>

I'll start by attaching `KNeighborsClassifier` to the preprocessing pipeline and plotting `1 - F1` error across `k = 1` to `30`. Using **F1 score** instead of raw accuracy is deliberate here—with ~84% of employees in the `No` class, a model could guess `No` every time and still look great on accuracy while completely failing to catch attrition.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

final_pipeline = Pipeline([("model",final_model), ("knn", KNeighborsClassifier())])

In [ ]:
from sklearn.metrics import f1_score

training_errors =[]
test_errors = []

k_values = range(1,31)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy = f1_score(df_train_y, train_pred)
    test_accuracy = f1_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("F1")
plt.grid(True)
plt.xticks(k_values)
plt.show()

Looking at the curve, error somewhat stabilizes around `k = 9` (going up to `k = 17` would add extra compute for only a very marginal improvement). However, because the error rate is high overall and smaller neighborhoods tend to be more sensitive to minority-class points, I'll also inspect `k = 3` directly via 5-fold cross-validation and on the test set before letting `GridSearchCV` search systematically.

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">5.2 K-Fold Cross-Validation on the Training Set</h3>
</div>

Evaluating the baseline KNN pipeline (`k = 3`) using 5-fold cross-validation on `df_train` to see the classification report and confusion matrix.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__n_neighbors=3)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">5.3 Baseline Evaluation on the Test Set</h3>
</div>

Fitting the baseline pipeline on the full training set and checking how it generalizes to the held-out test set.

In [ ]:
final_pipeline.set_params(knn__n_neighbors=3)

final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.plot()

<a id="section-6"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">06</span> &nbsp;|&nbsp; Hyperparameter Tuning &amp; Test Evaluation (Baseline KNN)
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Running GridSearchCV over neighbor counts, distance metrics, weighting schemes, and tree algorithms.</p>
</div>

First, I'll run a quick grid search just over odd values of `n_neighbors`, followed by a broader search across `weights`, `metric`, `leaf_size`, and `algorithm`—all scored on `f1` with 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(final_pipeline, param_grid=[{"knn__n_neighbors": [3,5,7,9,11,13,15,17]}], scoring="f1")

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)

In [ ]:
grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski", "manhattan", "euclidean"],
    "knn__leaf_size": [15, 30, 45, 60],
    "knn__algorithm": ["auto", "ball_tree", "kd_tree", "brute"]
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">6.1 Test Set Evaluation After Hyperparameter Tuning</h3>
</div>

Applying the best configuration found by `GridSearchCV` and evaluating it on `df_test`.

In [ ]:
final_pipeline.set_params(knn__n_neighbors=3, knn__algorithm="auto", knn__leaf_size=15, knn__metric="minkowski", knn__weights="uniform")

final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("K-fold Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.plot()

<a id="section-7"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">07</span> &nbsp;|&nbsp; Diagnosing Baseline Performance
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Understanding why raw KNN struggles on this dataset before moving to resampling and alternative models.</p>
</div>

Apparently, hyperparameter tuning did not have much impact on the baseline KNN model's performance. Despite achieving decent overall accuracy, its actual usefulness for the problem we're trying to solve—identifying employees likely to leave—is still mediocre and severely underfitting the minority class.

There are three main reasons why vanilla KNN struggles on this kind of dataset:

1. **Severe class imbalance:** There are considerably more employees staying than leaving (~5.2:1). In any local neighborhood, the majority class naturally outvotes the minority class.
2. **Lack of strong, clean separation:** As we saw in the EDA plots, the data is scattered with heavy overlap between the two target classes rather than forming tight, well-separated clusters.
3. **High dimensionality:** The dataset has a relatively large number of features, and almost none of them has a dominant individual relationship with `Attrition`. Distance metrics lose discriminative power as dimensions grow.

That doesn't mean the problem is hopeless, though. Next, I'll tackle the class imbalance directly using **SMOTENC** (which handles mixed numerical and categorical data properly) and benchmark **KNN**, **Random Forest**, and **Logistic Regression** on the resampled pipeline.

<a id="section-8"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">08</span> &nbsp;|&nbsp; Tackling Class Imbalance with SMOTENC (KNN)
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Oversampling the minority class inside an imbalanced-learn pipeline to avoid leakage during CV.</p>
</div>

To address the 5.2:1 class imbalance without leaking synthetic samples into validation folds, I'll use `imblearn.pipeline.Pipeline` with **SMOTENC** placed *after* imputation/scaling and *before* one-hot encoding. That way, SMOTENC can treat the categorical columns properly while oversampling only the training folds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

cat_cols = ["BusinessTravel" ,"JobRole", "MaritalStatus", "OverTime", "Gender", "Department", "EducationField"]

simple_imputer_num = SimpleImputer(strategy="median")
simple_imputer_cat = SimpleImputer(strategy="most_frequent")
standard_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(sparse_output=False)
ordinal_encoder = OrdinalEncoder()

cat_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat)])
num_pipeline = Pipeline([("simpleImputerNum", simple_imputer_num), ("standardScaler", standard_scaler)])

onehot_pipeline = Pipeline([ ("oneHotEncoder", one_hot_encoder)])
# ordinal_pipeline = Pipeline([("labelEncoder", ordinal_encoder)])

preprocessing_pipeline = ColumnTransformer([("num", num_pipeline, numerical_cols),
                                            ("cat", cat_pipeline, cat_cols),
                                            ])

encoding_pipeline = ColumnTransformer([("num", "passthrough", list(range(len(numerical_cols)))),
                                       ("onehot", one_hot_encoder, list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols)))),
                                      ])


final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("knn", KNeighborsClassifier())])

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">8.1 Plotting K Error Curves with F1 and Recall</h3>
</div>

Let's re-run the K sweep across odd values (`1` to `29`) using both **F1** and **Recall** to see how SMOTENC shifts the error curves.

In [ ]:
from sklearn.metrics import f1_score

training_errors =[]
test_errors = []

k_values = range(1,31, 2)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy = f1_score(df_train_y, train_pred)
    test_accuracy = f1_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("F1")
plt.grid(True)
plt.xticks(k_values)
plt.show()

In [ ]:
from sklearn.metrics import recall_score

training_errors =[]
test_errors = []

k_values = range(1,31, 2)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy =recall_score(df_train_y, train_pred)
    test_accuracy = recall_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("Recall")
plt.grid(True)
plt.xticks(k_values)
plt.show()

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">8.2 Cross-Validation &amp; Hyperparameter Tuning (KNN + SMOTENC)</h3>
</div>

After experimenting with the curves, `k = 13` with `manhattan` distance and `distance` weighting gives a much better balance of precision and recall on the minority class. Below, I'll evaluate `k = 13`, compare it against a smaller `k = 3` (where minority recall drops noticeably), and then run `GridSearchCV` to confirm the best settings.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__algorithm= 'auto', knn__leaf_size= 30, knn__metric= 'manhattan', knn__n_neighbors= 13, knn__weights= 'distance')
#after experimentation chose 13 kn neighbors as optimal, with good precision and recall for minority class

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

#### Comparing with a Smaller K (`k = 3`)

Testing a smaller neighborhood size here confirms that recall for the minority class drops significantly compared to `k = 13`.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__n_neighbors=3)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

#### Grid Search on KNN + SMOTENC

Running a quick search on `n_neighbors` followed by the full hyperparameter grid.

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(final_pipeline, param_grid=[{"knn__n_neighbors": [3,5,7,9,11,13,15,17]}], scoring="f1")

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)

In [ ]:
grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski", "manhattan", "euclidean"],
    "knn__leaf_size": [15, 30, 45, 60],
    "knn__algorithm": ["auto", "ball_tree", "kd_tree", "brute"]
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0 ,knn__algorithm= 'auto', knn__leaf_size= 30, knn__metric= 'manhattan', knn__n_neighbors= 13, knn__weights= 'distance')

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

Selecting the configuration above as it produces the strongest KNN results so far—around **0.36 precision** and **0.63 recall** on the minority class while still keeping solid numbers on the majority class.

<div style="border-bottom: 2px solid #38bdf8; padding-bottom: 4px; margin: 14px 0 8px 0;">
    <h3 style="margin: 0; font-size: 16px; font-weight: 600;">8.3 Test Set Evaluation (Tuned KNN + SMOTENC)</h3>
</div>

Fitting the tuned SMOTENC + KNN pipeline on the training set and checking final test performance.

In [ ]:
final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

<div style="border-left: 4px solid #f59e0b; background-color: rgba(56, 189, 248, 0.08); padding: 12px 16px; border-radius: 0 6px 6px 0; margin: 10px 0;">
    <strong>KNN + SMOTENC Takeaway</strong><br>
    SMOTENC clearly improved recall for the minority class, though the tradeoff is lower precision—false positives on the majority class went up noticeably. Overall, it's a significant step forward compared to raw KNN, but KNN is still struggling with the high dimensionality and class overlap. It makes a lot of sense to see how tree-based and linear models handle this exact same SMOTENC pipeline.
</div>

<a id="section-9"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">09</span> &nbsp;|&nbsp; Model Comparison: Random Forest Classifier
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Testing an ensemble of decision trees with SMOTENC and tuning depth/estimator count for recall.</p>
</div>

Next, I'll swap KNN out for a `RandomForestClassifier` inside the same SMOTENC pipeline. Tree ensembles handle non-linear interactions and high-dimensional one-hot features much more naturally than distance-based models. I'll evaluate a default 100-tree baseline via 5-fold CV and then run a grid search over `n_estimators` and `max_depth` optimizing for `recall`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("rf", RandomForestClassifier(max_depth=None, n_estimators=100))])

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report


kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
from sklearn.model_selection import GridSearchCV


grid = {
    "rf__n_estimators": [100, 200, 300, 500],
    "rf__max_depth": [None, 5, 10, 15, 20, 30],
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="recall" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

<a id="section-10"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">10</span> &nbsp;|&nbsp; Model Comparison: Logistic Regression
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Testing a regularized linear classifier with SMOTENC and balanced class weights.</p>
</div>

Finally, I'll test `LogisticRegression` with `class_weight="balanced"` on top of the SMOTENC pipeline. Because the features are already scaled and one-hot encoded, a linear model with L1/L2 regularization is a strong candidate when individual feature relationships with `Attrition` are weak but additive.

*Note: `solver="liblinear"` is specified so that both `l1` and `l2` penalties evaluate cleanly during `GridSearchCV` without solver mismatch warnings.*

In [ ]:
from sklearn.linear_model import LogisticRegression

final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("lr", LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
    random_state=42
))])

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

#### Grid Search on Logistic Regression

Tuning regularization strength (`C`), penalty type (`l1` vs. `l2`), and `max_iter` using 5-fold cross-validation scored on `f1`.

In [ ]:
from sklearn.model_selection import GridSearchCV


grid = {
    "lr__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "lr__penalty": ["l1", "l2"],
    # "lr__solver": ["lbfgs", "liblinear", "saga"],
    # "lr__class_weight": [None, "balanced"],
    # "lr__fit_intercept": [True, False],
    "lr__max_iter": [1000, 2000, 5000],
    # "lr__l1_ratio": []
    
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

#### Evaluating Tuned Logistic Regression (`C = 0.1`, `l2`)

Checking the cross-validated classification report and confusion matrix with the tuned regularization settings.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0, lr__C=  0.1, lr__max_iter=  1000, lr__penalty= 'l2')

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))

In [ ]:
import seaborn as sns
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

<a id="section-11"></a>
<div style="background: linear-gradient(90deg, #0f172a 0%, #1e293b 100%); border-left: 5px solid #38bdf8; padding: 14px 20px; border-radius: 6px; margin: 18px 0 10px 0;">
    <h2 style="margin: 0; color: #f8fafc; font-size: 20px; letter-spacing: 0.3px;">
        <span style="color: #38bdf8; font-weight: 700;">11</span> &nbsp;|&nbsp; Final Takeaways
    </h2>
    <p style="margin:4px 0 0 0; font-size:13.5px; color:#cbd5e1; font-weight:400;">Key lessons from benchmarking KNN, Random Forest, and Logistic Regression on imbalanced HR data.</p>
</div>

Looking across all the experiments in this notebook, a few practical takeaways stand out:

* **Accuracy is misleading on attrition data:** Because ~84% of employees stay, evaluating models on **F1** and **Recall** for the minority class (`Attrition = Yes`) is what actually tells us whether the model is doing its job.
* **Why baseline KNN fell short:** Without resampling, KNN got overwhelmed by the 5.2:1 class imbalance, high dimensionality, and overlapping feature distributions—hyperparameter tuning alone couldn't fix that.
* **SMOTENC helps, with a clear precision-recall tradeoff:** Oversampling inside `ImbPipeline` significantly boosted minority-class recall (~0.63 for tuned KNN), at the cost of more false positives on the majority class.
* **Model choice matters:** Moving from distance-based KNN to **Random Forest** and regularized **Logistic Regression** (`C = 0.1`, `l2`) gives much cleaner handling of weak, additive signals across mixed numerical and categorical features.

As a natural next step, experimenting with **decision threshold tuning** (adjusting the probability cutoff instead of relying on the default 0.5) or testing **SVM / Gradient Boosted Trees** on this same leak-free pipeline would be a great way to push minority-class F1 even further.